# Explore Qdrant Vector Database

This notebook provides a simple interface to connect to the local Qdrant instance (running via Docker) to verify that our OKF documents and their YAML metadata payloads were successfully ingested and indexed.

### 1. Imports and Setup

First, we import the official Qdrant client. Ensure your Docker container is running (`docker-compose up -d qdrant`).

In [ ]:
import os
from qdrant_client import QdrantClient

# Connect to the local Qdrant instance
client = QdrantClient(url="http://localhost:6333")

print("Connected to Qdrant!") 

: 

### 2. Verify Collections

Let's check if the `okf_knowledge` collection exists and fetch its basic statistics.

In [ ]:
collection_name = "okf_knowledge"

try:
    collection_info = client.get_collection(collection_name)

    print(f"Collection Name: {collection_name}")
    print(f"Total Vector Count (Points): {collection_info.points_count}")
    print(f"Vector Dimensions: {collection_info.config.params.vectors.size}")

except Exception as e:
    print(
        f"Error: Could not find collection '{collection_name}'. "
        f"Have you run the ingestion pipeline yet?\n\n{e}"
    )

### 3. Inspect a Vector and its OKF Payload

We can scroll through the collection to inspect a single point (vector). This verifies that the OKF YAML metadata (title, topics, trust_level, etc.) was successfully attached as a payload.

In [ ]:
try:
    records, next_page_offset = client.scroll(
        collection_name=collection_name,
        limit=1,
        with_payload=True,
        with_vectors=False
    )

    if records:
        print("--- Sample Vector Payload ---")

        sample_payload = records[0].payload

        for key, value in sample_payload.items():
            if key != "_node_content":
                print(f"{key.capitalize()}: {value}")

        if "_node_content" in sample_payload:
            import json

            node_data = json.loads(sample_payload["_node_content"])
            print(
                f"\nText Snippet:\n{node_data.get('text', '')[:200]}..."
            )

    else:
        print("Collection is empty. No points to display.")

except Exception as e:
    print(f"Error fetching records: {e}")

### 4. Test Metadata Filtering (The Power of OKF)

This demonstrates metadata filtering by retrieving only documents whose `trust_level` is set to **High**.

In [ ]:
from qdrant_client.http.models import (
    Filter,
    FieldCondition,
    MatchValue,
)

try:
    okf_filter = Filter(
        must=[
            FieldCondition(
                key="trust_level",
                match=MatchValue(value="High"),
            )
        ]
    )

    filtered_records, _ = client.scroll(
        collection_name=collection_name,
        scroll_filter=okf_filter,
        limit=3,
        with_payload=True,
        with_vectors=False,
    )

    print(
        f"Found {len(filtered_records)} chunks marked with 'High' trust level."
    )

    for i, record in enumerate(filtered_records, start=1):
        print(f"Result {i}: {record.payload.get('title', 'Unknown')}")

except Exception as e:
    print(f"Error testing filter: {e}")